In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from underthesea import sent_tokenize
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bert_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sent_emb = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(sent_emb)

def split_sentences(text):
    if not isinstance(text, str):
        return []
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def get_top_k_by_ratio(num_sentences, ratio=0.6):
    return max(1, int(num_sentences * ratio))

Device: cuda


In [2]:
MODEL_PATH = r"E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_TX\phobert_summary_model.pth"

print(f"Đang load model từ: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

encoder = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
model = PhoBERTSUM(encoder).to(DEVICE)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✅ Model đã được load thành công")
print(f"Best params: {checkpoint.get('best_params', 'N/A')}")
print(f"Test accuracy: {checkpoint.get('test_acc', 'N/A'):.4f}" if 'test_acc' in checkpoint else "Test accuracy: N/A")

Đang load model từ: E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_TX\phobert_summary_model.pth


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\minhv\AppData\Local\Temp\ipykernel_5644\3212734973.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will n

✅ Model đã được load thành công
Best params: {'learning_rate': 3.868107453864181e-05, 'batch_size': 32, 'max_len': 128, 'warmup_steps': 213}
Test accuracy: 0.7119


In [3]:
@torch.no_grad()
def extractive_summary(text, ratio=0.6, max_len=128):
    sentences = split_sentences(text)
    n = len(sentences)

    if n == 0:
        return ""

    if n == 1:
        return sentences[0]

    top_k = get_top_k_by_ratio(n, ratio)

    encoded = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    
    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)
    
    scores = model(input_ids, attention_mask)
    scores = scores.squeeze().cpu()

    top_idx = torch.topk(scores, min(top_k, n)).indices.tolist()
    
    if not isinstance(top_idx, list):
        top_idx = [top_idx]

    top_idx.sort()
    summary = " ".join(sentences[i] for i in top_idx)
    return summary

In [4]:
rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def evaluate_summary(candidate, reference):
    if not candidate or not reference:
        return {
            'f1': 0.0,
            'rouge1': 0.0,
            'rougel': 0.0,
            'bertscore': 0.0
        }
    
    rouge_scores = rouge_scorer_obj.score(reference, candidate)
    rouge1 = rouge_scores['rouge1'].fmeasure
    rougel = rouge_scores['rougeL'].fmeasure
    
    try:
        P, R, F1_bert = bert_score([candidate], [reference], lang='vi', verbose=False)
        bertscore = F1_bert.item()
    except:
        bertscore = 0.0
    
    candidate_tokens = set(candidate.lower().split())
    reference_tokens = set(reference.lower().split())
    if len(reference_tokens) == 0:
        f1 = 0.0
    else:
        intersection = candidate_tokens & reference_tokens
        precision = len(intersection) / len(candidate_tokens) if len(candidate_tokens) > 0 else 0
        recall = len(intersection) / len(reference_tokens) if len(reference_tokens) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'f1': f1,
        'rouge1': rouge1,
        'rougel': rougel,
        'bertscore': bertscore
    }

In [9]:
test_texts = [
    {
        "content": """Phong cảnh quê Bác 
Đường vô xứ Nghệ quanh quanh Non xanh nước biếc như tranh họa đồ. Câu hát của người xưa cứ ngân nga trong tâm trí chúng tôi trên con đường chúng tôi đi về quê Bác. Giữa khung cảnh "non xanh nước biếc", chúng tôi mãi mê nhìn những cánh đồng chiêm mơn mởn, những chiếc cầu sắt mới tình, duyên dáng, những mái trường, mái nhà tươi rói bên cạnh những rặng tre non. Phong cảnh vùng này quả thật là đẹp. Đứng trên núi Chung, nhìn sang bên trái là dòng sông Lam uốn khúc theo dây núi Thiên Nhẫn. Mặt sông hất ánh nắng chiều thành một đường quanh co trắng xóa. Nhìn sang bên phải là dây núi Trác nối liền với dãy núi Đại Huệ xa xa. Trước mặt chúng tôi, giữa hai dãy núi, là nhà Bác với cánh đồng quê Bác. Nhìn xuống cánh đồng, có đủ các màu xanh: xanh pha vàng của ruộng mía, xanh rất mượt mà của lúa chiêm đang thì con gái, xanh đậm của những rặng tre; đây đó một vài cây phi lao xanh biếc và rất nhiều màu xanh khác nữa. Cả cánh đồng thu gọn trong tầm mắt, làng nối làng, ruộng tiếp ruộng. Cuộc sống ở đây có một cái gì mặn mà, ấm áp.""",
        "reference": """Phong cảnh quê Bác 
Đường vô xứ Nghệ quanh quanh Non xanh nước biếc như tranh họa đồ. Câu hát của người xưa cứ ngân nga trong tâm trí chúng tôi trên con đường chúng tôi đi về quê Bác. Phong cảnh vùng này quả thật là đẹp. Mặt sông hất ánh nắng chiều thành một đường quanh co trắng xóa. Nhìn sang bên phải là dây núi Trác nối liền với dãy núi Đại Huệ xa xa. Trước mặt chúng tôi, giữa hai dãy núi, là nhà Bác với cánh đồng quê Bác."""
    },
    {
        "content": """Lòng yêu nước ban đầu là lòng yêu những vật tầm thường nhất. Lòng yêu nước chính là tình yêu quê hương đất tổ nơi mình sinh ra và lớn lên. Lòng yêu nước có thể là yêu lũy tre xanh bao quanh làng, yêu dòng sông chảy trước nhà, yêu chân ruộng thơm mùi gốc rạ, yêu con cò đứng khoan thai trên đồng bên dáng mẹ còng lưng làm cỏ lúa… Tình cảm ấy đơn giản, gần gũi và nằm ngay trong lời ăn tiếng nói hằng ngày của mỗi người. Mỗi chúng ta từ lúc sinh ra cho tới lúc lớn khôn và trưởng thành thì gia đình là nơi nuôi dưỡng, dạy dỗ. Đó là nơi chúng ta cần yêu thương đầu tiên. Mai này chúng ta lớn lên có trường học, xã hội, những người bạn xung quanh. Chúng ta cần phải san sẻ tình yêu thương của mình cho tất cả mọi người nếu có thể. Đôi khi lòng yêu nước chỉ là tình cảm đơn giản, bình dị như vậy nhưng lại có ý nghĩa rất lớn.""",
        "reference": """Lòng yêu nước ban đầu là lòng yêu những vật tầm thường nhất. Lòng yêu nước chính là tình yêu quê hương đất tổ nơi mình sinh ra và lớn lên. Đôi khi lòng yêu nước chỉ là tình cảm đơn giản, bình dị như vậy nhưng lại có ý nghĩa rất lớn."""
    }
]

print("=" * 60)
print("TEST MODEL VỚI CÁC VÍ DỤ")
print("=" * 60)

for i, test_case in enumerate(test_texts, 1):
    print(f"\n[TEST {i}]")
    print(f"Văn bản gốc: {test_case['content']}")
    
    summary = extractive_summary(test_case['content'], ratio=0.6)
    print(f"\nTóm tắt được tạo:")
    print(f"{summary}")
    
    print(f"\nTóm tắt tham chiếu:")
    print(f"{test_case['reference']}")
    
    scores = evaluate_summary(summary, test_case['reference'])
    print(f"\nĐiểm đánh giá:")
    print(f"  F1-Score: {scores['f1']:.4f}")
    print(f"  ROUGE-1: {scores['rouge1']:.4f}")
    print(f"  ROUGE-L: {scores['rougel']:.4f}")
    print(f"  BERTScore: {scores['bertscore']:.4f}")
    print("-" * 60)

TEST MODEL VỚI CÁC VÍ DỤ

[TEST 1]
Văn bản gốc: Phong cảnh quê Bác 
Đường vô xứ Nghệ quanh quanh Non xanh nước biếc như tranh họa đồ. Câu hát của người xưa cứ ngân nga trong tâm trí chúng tôi trên con đường chúng tôi đi về quê Bác. Giữa khung cảnh "non xanh nước biếc", chúng tôi mãi mê nhìn những cánh đồng chiêm mơn mởn, những chiếc cầu sắt mới tình, duyên dáng, những mái trường, mái nhà tươi rói bên cạnh những rặng tre non. Phong cảnh vùng này quả thật là đẹp. Đứng trên núi Chung, nhìn sang bên trái là dòng sông Lam uốn khúc theo dây núi Thiên Nhẫn. Mặt sông hất ánh nắng chiều thành một đường quanh co trắng xóa. Nhìn sang bên phải là dây núi Trác nối liền với dãy núi Đại Huệ xa xa. Trước mặt chúng tôi, giữa hai dãy núi, là nhà Bác với cánh đồng quê Bác. Nhìn xuống cánh đồng, có đủ các màu xanh: xanh pha vàng của ruộng mía, xanh rất mượt mà của lúa chiêm đang thì con gái, xanh đậm của những rặng tre; đây đó một vài cây phi lao xanh biếc và rất nhiều màu xanh khác nữa. Cả cánh đồng thu 

In [ ]:
TEST_DATA_PATH = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_TX_with_summary.xlsx"

try:
    df_test = pd.read_excel(TEST_DATA_PATH)
    
    if 'content' in df_test.columns and 'summary' in df_test.columns:
        print(f"\nĐang test với {len(df_test)} mẫu từ file dữ liệu...")
        
        results = []
        for idx, row in df_test.head(20).iterrows():
            if pd.isna(row['content']) or pd.isna(row.get('summary', '')):
                continue
                
            content = str(row['content'])
            reference = str(row.get('summary', ''))
            
            summary = extractive_summary(content, ratio=0.6)
            scores = evaluate_summary(summary, reference)
            
            results.append({
                'doc_id': idx,
                'f1': scores['f1'],
                'rouge1': scores['rouge1'],
                'rougel': scores['rougel'],
                'bertscore': scores['bertscore']
            })
        
        if results:
            df_results = pd.DataFrame(results)
            print(f"\n{'='*60}")
            print("KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET")
            print(f"{'='*60}")
            print(f"Số mẫu đã test: {len(results)}")
            print(f"\nĐiểm trung bình:")
            print(f"  F1-Score: {df_results['f1'].mean():.4f}")
            print(f"  ROUGE-1: {df_results['rouge1'].mean():.4f}")
            print(f"  ROUGE-L: {df_results['rougel'].mean():.4f}")
            print(f"  BERTScore: {df_results['bertscore'].mean():.4f}")
    else:
        print("File không có cột 'content' hoặc 'summary'")
        
except FileNotFoundError:
    print(f"Không tìm thấy file test data tại: {TEST_DATA_PATH}")
except Exception as e:
    print(f"Lỗi khi đọc file test: {e}")